## Import and load

In [ ]:
import pickle
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from pathlib import Path
import replicate
from PIL import Image  # Import Image for opening images
import matplotlib.pyplot as plt  # Import matplotlib for displaying images

## Load embeddings

In [ ]:
# Load the embeddings for cropped objects
OBJECT_EMBEDDINGS_FILE = "../app/embeddings/object_embeddings.pkl"
FRAME_EMBEDDINGS_FILE = "../app/embeddings/frame_embeddings.pkl"

with open(OBJECT_EMBEDDINGS_FILE, "rb") as f:
    object_embeddings = pickle.load(f)

with open(FRAME_EMBEDDINGS_FILE, "rb") as f:
    frame_embeddings = pickle.load(f)

print(f"Loaded {len(object_embeddings)} object embeddings.")
print(f"Loaded {len(frame_embeddings)} frame embeddings.")

Loaded 4330 object embeddings.
Loaded 613 frame embeddings.


## Define function to embed query

In [8]:
# Load Replicate API token
import os
from dotenv import load_dotenv

load_dotenv()
REPLICATE_API_TOKEN = os.getenv("REPLICATE_API_TOKEN")

if not REPLICATE_API_TOKEN:
    raise RuntimeError("Missing Replicate API token. Add it to .env as REPLICATE_API_TOKEN=xxx.")

# Create Replicate client
client = replicate.Client(api_token=REPLICATE_API_TOKEN)

def embed_text_query(query):
    """
    Compute the embedding for a text query using CLIP.
    """
    output = client.run(
        "openai/clip",
        input={"text": query, "task": "embed"},
    )
    return np.array(output["embedding"])

## Define semantic search function

In [13]:
def semantic_search(query, object_embeddings, frame_embeddings, top_k=5):
    """
    Perform semantic search on both object and frame embeddings using a text query.

    Args:
        query (str): The text query.
        object_embeddings (dict): A dictionary of object embeddings {path: embedding}.
        frame_embeddings (dict): A dictionary of frame embeddings {path: embedding}.
        top_k (int): Number of top results to return.

    Returns:
        dict: Top-k results for objects and frames as {"objects": [...], "frames": [...]}.
    """
    # Embed the query
    query_embedding = embed_text_query(query).reshape(1, -1)

    # Normalize the embeddings and the query
    # object_matrix = np.array(list(object_embeddings.values()))
    # object_matrix = normalize(object_matrix, axis=1)
    frame_matrix = np.array(list(frame_embeddings.values()))
    frame_matrix = normalize(frame_matrix, axis=1)
    query_embedding = normalize(query_embedding, axis=1)

    # Compute cosine similarity for objects
    # object_paths = list(object_embeddings.keys())
    # object_similarities = cosine_similarity(query_embedding, object_matrix)[0]
    # object_top_indices = np.argsort(object_similarities)[::-1][:top_k]
    # object_results = [(object_paths[i], object_similarities[i]) for i in object_top_indices]

    # Compute cosine similarity for frames
    frame_paths = list(frame_embeddings.keys())
    frame_similarities = cosine_similarity(query_embedding, frame_matrix)[0]
    frame_top_indices = np.argsort(frame_similarities)[::-1][:top_k]
    frame_results = [(frame_paths[i], frame_similarities[i]) for i in frame_top_indices]

    return {"objects": [], "frames": frame_results}  # Return only frame results for debugging

In [14]:
def show_search_results(results, base_dir="app/cropped_objects", frames_dir="app/output_frames"):
    """
    Displays the top search results as thumbnails in a gallery with three columns.
    Each result includes the cropped object and its corresponding full frame side by side.
    
    Args:
        results (list): List of tuples (path, similarity).
        base_dir (str): Base directory where cropped images are stored.
        frames_dir (str): Base directory where full frames are stored.
    """
    base_dir = Path(base_dir).resolve()  # Convert base_dir to an absolute path
    frames_dir = Path(frames_dir).resolve()  # Convert frames_dir to an absolute path
    num_results = len(results)
    num_columns = 2  # Two columns: one for cropped object, one for full frame
    num_rows = (num_results + num_columns - 1) // num_columns  # Calculate the number of rows

    # Create a figure for the gallery
    fig, axes = plt.subplots(num_rows, num_columns, figsize=(15, 5 * num_rows))
    axes = axes.flatten()  # Flatten the axes array for easy iteration

    for i, (path, similarity) in enumerate(results):
        full_path = base_dir / path
        if not full_path.exists():
            continue

        try:
            # Open the cropped object image
            cropped_img = Image.open(full_path)

            # Extract the video folder and frame name from the cropped object path
            video_folder = "_".join(path.split("_")[:2])  # Extract "video_1"
            frame_name = "_".join(path.split("_")[:4]) + ".jpg"  # Include the full frame number
            full_frame_path = frames_dir / video_folder / frame_name  # Add the video folder to the path

            if not full_frame_path.exists():
                continue

            # Open the full frame image
            full_frame_img = Image.open(full_frame_path)

            # Display the cropped object in the gallery
            axes[2 * i].imshow(cropped_img)
            axes[2 * i].axis("off")
            axes[2 * i].set_title(f"Cropped: {path}\nSimilarity: {similarity:.4f}", fontsize=10)

            # Display the full frame in the gallery
            axes[2 * i + 1].imshow(full_frame_img)
            axes[2 * i + 1].axis("off")
            axes[2 * i + 1].set_title(f"Full Frame: {frame_name}", fontsize=10)

        except Exception as e:
            pass  # Suppress errors and continue processing

    # Hide any unused subplots
    for j in range(2 * len(results), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

## Run query

In [15]:
# Run a semantic search query
query = "police line"
top_k = 20

results = semantic_search(query, object_embeddings, frame_embeddings, top_k=top_k)

# Visualize the top object results
print("Top object results:")
show_search_results(results["objects"], base_dir="../app/cropped_objects", frames_dir="../app/output_frames")

# Visualize the top frame results
print("Top frame results:")
show_search_results(results["frames"], base_dir="../app/output_frames", frames_dir="../app/output_frames")

Top object results:


ValueError: Number of rows must be a positive integer, not 0

<Figure size 1500x0 with 0 Axes>